# Structured Streaming: process never-ending data with the batch API

_Apache Spark_

**Treat a live stream as a table that keeps growing, run it in tiny micro-batches, and write almost the same code you'd write for batch.**

Most data work assumes the data sits still: a file, a table, a frame you read once and crunch.
       Streaming data does not sit still &mdash; it keeps arriving, forever: clicks, payments,
       sensor readings, log lines, one after another with no end. The data is unbounded (it has no last
       row). You cannot wait for "all of it" before you start, because "all of it" never happens.

       Spark's Structured Streaming has one beautiful idea that makes this easy: treat the stream as
       a table that never stops growing. Every new event is a new row appended to the bottom of an
       unbounded table. Your job is just a query over that table &mdash; the same select,
       filter, groupBy you would write for a normal Spark DataFrame. Spark re-runs that
       query, again and again, on the newly-arrived rows.

---

This notebook builds a streaming query up one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PySpark (Structured Streaming)

We'll build a streaming query that counts events per time window. We do it in four steps: (1) start Spark and open a streaming **source**, (2) write the windowed **transform** — the exact same code you'd write for batch, (3) start the **sink** that emits results, (4) see how the identical logic reads as a bounded batch job.

### Step 1 — Start Spark and open a streaming source

A streaming query starts from a `readStream` source instead of a static `read`. Here we use the built-in **rate** source, which emits a steady stream of rows (a `timestamp` and a `value`) at a fixed rate — handy for demos with no Kafka cluster. In production this is the only line that changes: you'd swap in `spark.readStream.format("kafka")...` instead.

In [ ]:
# Colab: !pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col

spark = (SparkSession.builder
         .master("local[*]").appName("streaming-demo")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

# SOURCE: the built-in 'rate' stream (timestamp + value, N rows/sec).
# In production this line is the only thing that changes, e.g.:
#   spark.readStream.format("kafka").option("subscribe","clicks").load()
events = (spark.readStream
          .format("rate")
          .option("rowsPerSecond", 20)   # 20 events per second
          .load())                       # columns: timestamp (event time), value

### Step 2 — Write the windowed transform (identical to batch)

This is the whole point of Structured Streaming: the transform is **byte-for-byte the same** as the batch API. We group events into tumbling 10-second windows by their event time and count them. The `withWatermark` tells Spark to wait up to 30 seconds for late-arriving events before it finalizes and drops a window's state — that's what keeps memory bounded.

In [ ]:
# TRANSFORM: windowed count by EVENT TIME, with a watermark.
# >>> This block is byte-for-byte the same as the BATCH API <<<
windowed = (events
            .withWatermark("timestamp", "30 seconds")          # wait up to 30s for late events
            .groupBy(window(col("timestamp"), "10 seconds"))   # tumbling 10s windows
            .count())

### Step 3 — Start the sink

`writeStream ... .start()` is what actually launches the continuous query. We write to the **console** in `update` mode — because a running count keeps changing, we emit only the windows whose count changed this batch (not `append`, which is for final-only rows). The `checkpointLocation` durably records read progress and state so a restart resumes exactly-once, and `trigger` fires one micro-batch every 5 seconds.

In [ ]:
# SINK: console, UPDATE mode (emit windows whose count changed), checkpointed.
query = (windowed.writeStream
         .outputMode("update")                 # running counts change -> 'update', not 'append'
         .format("console")
         .option("truncate", False)
         .option("checkpointLocation", "/tmp/ckpt_stream")  # exactly-once on restart
         .trigger(processingTime="5 seconds")  # one micro-batch every 5s
         .start())

# query.awaitTermination(30)   # let it run ~30s in Colab...
# query.stop()                 # ...then stop it.

### Step 4 — The batch twin

To drive the point home: the same `groupBy / window / count` works on bounded data with a plain `read` and `.show()`. The streaming version above and the batch version below share the *exact* transform — that's the promise of Structured Streaming, write batch logic and let Spark run it continuously.

In [ ]:
# The batch twin (same logic, bounded data):
#   static = spark.read.format("rate")...   # or spark.read.parquet(path)
#   static.groupBy(window(col("timestamp"), "10 seconds")).count().show()
# Same groupBy/window/count -> that is the whole point of Structured Streaming.

## Visualize the data & results

_What does a windowed streaming aggregation actually output — event counts per tumbling time window — and how many events does a 5-second watermark accept as on-time vs drop as late?_

### Step 1 — Synthesize a 60-second event stream

To see what a windowed aggregation outputs without running a live stream for a minute, we generate 320 synthetic events. Their event-time second is drawn from a bell curve peaking mid-minute (so the rate rises then falls), and each event gets a random arrival delay — the gap between when it happened and when it reaches Spark.

In [ ]:
import numpy as np
import pandas as pd

# Synthetic 60s event stream: 320 events, rate peaks mid-window.
rng = np.random.RandomState(7)
start = pd.Timestamp("2026-06-21 10:00:00")

# Bell-shaped per-second weights so the event rate rises then falls.
raw_weights = np.exp(-((np.arange(60) - 30) ** 2) / (2 * 14 ** 2))
weights = raw_weights / raw_weights.sum()

secs  = np.sort(rng.choice(np.arange(60), size=320, p=weights))   # event-time second
delay = np.abs(rng.normal(2.0, 3.0, size=320))                    # arrival delay (s)

ev = pd.DataFrame({"event_time": start + pd.to_timedelta(secs, unit="s"),
                   "delay": delay})

### Step 2 — Bucket events into tumbling windows

This is exactly what `groupBy(window(...))` does: assign each event to the 10-second window it falls in (by flooring its event time), then count per window. The six window counts sum back to all 320 events.

In [ ]:
# Tumbling 10s windows by EVENT TIME (what groupBy(window(...)) does).
ev["win"] = ev["event_time"].dt.floor("10s")
counts = ev.groupby("win").size()
print("per-window counts:", counts.tolist())     # [23, 50, 87, 81, 59, 20]
print("total events:", int(counts.sum()))        # 320

### Step 3 — Split on-time vs late with a watermark

A 5-second watermark means any event that arrives more than 5 seconds after its event time is **late** and gets dropped once its window is finalized. Splitting the events on `delay > 5` shows how many each window keeps versus loses — a concrete picture of what a watermark trades away for bounded state.

In [ ]:
# 5s watermark: events delayed > 5s are 'late' (dropped after finalize).
ev["late"] = ev["delay"] > 5.0
ontime = (~ev["late"]).groupby(ev["win"]).sum().astype(int)
late   = ev["late"].groupby(ev["win"]).sum().astype(int)
print("on-time per window:", ontime.tolist())     # [20, 42, 76, 70, 50, 19]
print("late per window   :", late.tolist())       # [3, 8, 11, 11, 9, 1]
print("total on-time", int(ontime.sum()),         # 277
      " total late", int(late.sum()))             # 43

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You run events.groupBy(window("event_time","1 minute")).count() as a streaming query with no withWatermark. It works for an hour, then the driver dies with an out-of-memory error. What happened, and what's the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that each 1-minute window keeps a running count in Spark's state store. — _An aggregation needs to remember the current count per group (here, per window) to update it as new rows arrive._
- Realize that without a watermark, Spark cannot know a window is "done", so it never drops any window's state. — _A late event could in principle still belong to any past window, so all windows must be kept open forever._
- Add events.withWatermark("event_time","10 minutes").groupBy(window("event_time","1 minute")).count(). — _The watermark lets Spark retire windows older than the lateness horizon, freeing their state and bounding memory._

**Answer:** Unbounded state growth. Every 1-minute window holds aggregation state, and with no watermark Spark must keep every window's state indefinitely (a late event might belong to any of them), so state &mdash; and memory &mdash; grows without bound until the driver dies. The fix is a watermark: withWatermark("event_time","10 minutes") tells Spark to finalize and drop windows once the watermark passes their end, keeping only the windows within the lateness horizon. Size the lateness to how late your events realistically arrive.

</details>

**Problem 2.** You want a live count of clicks per user, updated continuously, written to the console. You try ...groupBy("user").count().writeStream.outputMode("append").format("console").start() and Spark refuses to start the query. Why, and which output mode should you use?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the query as a running aggregation: each user's count changes whenever a new click for that user arrives. — _The result row for a user is not final after one batch — later batches keep updating it._
- Recall that append only emits rows that will never change again. — _Spark cannot emit a row in append mode while that row can still be updated, so an unbounded running aggregation has no append-able rows — Spark rejects it._
- Switch to outputMode("update") (or complete if the number of users is small). — _Update emits exactly the user rows whose counts changed this batch — the natural fit for a live running aggregate._

**Answer:** append is the wrong mode. Append only outputs rows that are final, but a running per-user count keeps changing as new clicks arrive, so there are never any final rows to append &mdash; Spark rejects the query. Use update, which emits just the rows whose counts changed each micro-batch (ideal for a live dashboard). complete also works but re-emits the entire table every batch, so only use it when the number of distinct users (groups) stays small.

</details>

**Problem 3.** A teammate asks why your streaming feature pipeline sets a checkpointLocation, since "it ran fine in testing without one." What do you tell them?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Point out that a streaming job runs 24/7 and will eventually restart — deploys, crashes, machine loss. — _Unlike a batch job that finishes, a stream's lifetime is effectively forever, so a restart is a certainty, not an edge case._
- Explain that the checkpoint stores read progress (offsets) and aggregation state durably. — _On restart Spark reads the checkpoint to resume exactly where it stopped instead of re-reading from the start or skipping ahead._
- Conclude that the checkpoint is what gives exactly-once processing — no lost events, no duplicates. — _Without it, a restart re-processes already-handled events (duplicates) or skips unread ones (loss), corrupting downstream features._

**Answer:** It "ran fine" only because it never restarted. A streaming job runs continuously and will restart eventually (deploy, crash, node loss). The checkpoint durably records how far it has read from the source plus the current aggregation state, so on restart Spark resumes exactly where it left off &mdash; this is what delivers exactly-once processing. Without a checkpoint a restart re-reads already-processed data (duplicates) or skips unread data (loss), quietly corrupting the features downstream. Always set a durable, per-query checkpointLocation.

</details>